In [ ]:
import sys
from pathlib import Path
import pandas as pd

# ---------------------------------------------------
# 1) Make sure Python can find the bkt package
# ---------------------------------------------------
# Option A: if you extracted the zip and have a folder called "bkt_package"
# Option B: if you copied only the "bkt" folder into your project root,
# you can comment the previous line and use your normal imports directly.

from bkt.data.schemas import ColumnMapping
from bkt.data.preprocessing import prepare_bkt_dataframe
from bkt.data.sequence_builder import build_sequences_for_skill, build_all_sequences
from bkt.training.trainer import BKTTrainer
from bkt.inference.predictor import BKTModel
from bkt.inference.mastery_updater import StudentMasteryTracker
from bkt.evaluation.metrics import evaluate_sequences
from bkt.persistence.save_load import save_skill_models_json

# ---------------------------------------------------
# 2) Use your dataframe
# ---------------------------------------------------
# If train_bkt is already loaded, keep this:
df_raw = train_bkt.copy()

# If you want to load from file instead, use something like:
# df_raw = pd.read_csv("train_bkt.csv")

print("Raw shape:", df_raw.shape)
print(df_raw.head())

# ---------------------------------------------------
# 3) Define the mapping for your columns
# ---------------------------------------------------
mapping = ColumnMapping(
    student_id="user_id",
    skill_name="skill_name",
    correct="correct",
    order_id="order_id",
)

# ---------------------------------------------------
# 4) Clean and prepare the dataframe
# ---------------------------------------------------
df = prepare_bkt_dataframe(
    df_raw,
    mapping=mapping,
    drop_na=True,
    remove_duplicates=True,
)

print("\nPrepared shape:", df.shape)
print(df.head())

print("\nUnique students:", df["user_id"].nunique())
print("Unique skills:", df["skill_name"].nunique())
print("\nCorrect column distribution:")
print(df["correct"].value_counts(dropna=False).sort_index())

# ---------------------------------------------------
# 5) Basic sequence diagnostics
# ---------------------------------------------------
seqs_by_skill = build_all_sequences(df, mapping=mapping, min_sequence_length=1)

skill_summary = []
for skill, seqs in seqs_by_skill.items():
    n_students = len(seqs)
    n_obs = sum(len(s.observations) for s in seqs)
    avg_len = n_obs / n_students if n_students > 0 else 0
    skill_summary.append((skill, n_students, n_obs, avg_len))

skill_summary_df = pd.DataFrame(
    skill_summary,
    columns=["skill_name", "n_students", "n_obs", "avg_seq_len"]
).sort_values("n_obs", ascending=False)

print("\nTop skills by number of observations:")
print(skill_summary_df.head(20))

# ---------------------------------------------------
# 6) Fit one skill first
# ---------------------------------------------------
one_skill = "normalize_negative_sign"   # change this if needed

seqs_one = build_sequences_for_skill(
    df,
    mapping=mapping,
    skill_name=one_skill,
    min_sequence_length=1,
)

print(f"\nSkill selected: {one_skill}")
print("Number of student sequences:", len(seqs_one))
print("Total observations:", sum(len(s.observations) for s in seqs_one))

trainer = BKTTrainer(
    max_iter=100,
    tol=1e-6,
    fit_forget=False,   # start with standard BKT
)

single_result = trainer.fit_skill(seqs_one)

print("\nSingle-skill fitted parameters:")
print(single_result.params)
print("Log-likelihood:", single_result.log_likelihood)
print("Iterations:", single_result.iterations)
print("Converged:", single_result.converged)

# ---------------------------------------------------
# 7) Evaluate that one skill
# ---------------------------------------------------
single_model = BKTModel(single_result.params)
single_metrics = evaluate_sequences(single_model, seqs_one)

print("\nSingle-skill evaluation:")
for k, v in single_metrics.items():
    print(f"{k}: {v}")

# ---------------------------------------------------
# 8) Inspect one student's forward trace
# ---------------------------------------------------
example_seq = seqs_one[0]
trace = single_model.run_sequence(example_seq.observations)

trace_rows = []
for step in trace.steps:
    trace_rows.append({
        "step_index": step.step_index,
        "observation": step.observation,
        "prior_known": step.prior_known,
        "predicted_correct": step.predicted_correct,
        "posterior_known": step.posterior_known,
        "next_prior_known": step.next_prior_known,
        "step_likelihood": step.step_likelihood,
    })

trace_df = pd.DataFrame(trace_rows)

print("\nExample student:", example_seq.student_id)
print(trace_df.head(20))
print("Final mastery:", trace.final_mastery)

# ---------------------------------------------------
# 9) Fit all skills
# ---------------------------------------------------
all_result = trainer.fit_from_dataframe(
    df,
    mapping=mapping,
    min_sequence_length=1,
)

params_table = []
for skill, fit_result in all_result.skill_results.items():
    p = fit_result.params
    params_table.append({
        "skill_name": skill,
        "p_init": p.p_init,
        "p_learn": p.p_learn,
        "p_guess": p.p_guess,
        "p_slip": p.p_slip,
        "p_forget": p.p_forget,
        "log_likelihood": fit_result.log_likelihood,
        "iterations": fit_result.iterations,
        "converged": fit_result.converged,
    })

params_df = pd.DataFrame(params_table).sort_values("skill_name").reset_index(drop=True)

print("\nAll fitted skills:")
print(params_df.head(20))

# ---------------------------------------------------
# 10) Evaluate all skills separately
# ---------------------------------------------------
eval_rows = []
for skill, seqs in seqs_by_skill.items():
    if skill not in all_result.params_by_skill:
        continue

    model = BKTModel(all_result.params_by_skill[skill])
    metrics = evaluate_sequences(model, seqs)

    eval_rows.append({
        "skill_name": skill,
        **metrics
    })

eval_df = pd.DataFrame(eval_rows).sort_values("n_obs", ascending=False).reset_index(drop=True)

print("\nEvaluation by skill:")
print(eval_df.head(20))

# ---------------------------------------------------
# 11) Save fitted parameters
# ---------------------------------------------------
save_path = save_skill_models_json(
    all_result.params_by_skill,
    "bkt_skill_models.json"
)
print("\nSaved model parameters to:", save_path)

# ---------------------------------------------------
# 12) Simulate online student tracking
# ---------------------------------------------------
tracker = StudentMasteryTracker(all_result.params_by_skill)

student_id = str(df.iloc[0]["user_id"])
skill_name = str(df.iloc[0]["skill_name"])
obs = int(df.iloc[0]["correct"])

before = tracker.get_mastery(student_id, skill_name)
pred_before = tracker.predict_correct(student_id, skill_name)
after = tracker.update(student_id, skill_name, obs)

print("\nOnline tracking example:")
print("student_id:", student_id)
print("skill_name:", skill_name)
print("observation:", obs)
print("mastery before:", before)
print("predicted correct before:", pred_before)
print("mastery after:", after)

ImportError: attempted relative import with no known parent package